### Setup für alle Aufgaben

In [1]:
from pymongo import MongoClient
from datetime import datetime
from bson import ObjectId

# Verbindung zur lokalen MongoDB
client = MongoClient('localhost', 27017)

# Datenbank "university" auswählen
db = client["university"]

# Collections
col = db["studenten"]

# Vor jeder Aufgabe ggf. alte Daten löschen (optional)
#col.delete_many(["_id": ""])

#### Aufgabe 1: einfacher Insert

In [2]:
col.insert_one({
    'name': 'Anna',
    'alter': 24,
    'isStudent': True
    })

InsertOneResult(ObjectId('69c0ec0fb31cfe390202dd14'), acknowledged=True)

#### Aufgabe 2: Mehrere Dokumente einfügen

In [3]:
col.insert_many([
    {"name": "Jonas", "skills": ["Python", "MongoDB"], "alter": 24},
    {"name": "Lea", "skills": ["C++", "ML"], "alter": 26}
])

InsertManyResult([ObjectId('69c0ec11b31cfe390202dd15'), ObjectId('69c0ec11b31cfe390202dd16')], acknowledged=True)

#### Aufgabe 3: Dokumente abfragen

In [4]:
query = col.find({'alter': {'$gt': 23}})
print(query)
for doc in query:
    print(doc)
    print(doc['name'], doc['alter'])

{'_id': ObjectId('69ae8d8f85fca114abbd6128'), 'name': 'Anna', 'alter': 24, 'isStudent': True, 'eingeschrieben_am': datetime.datetime(2026, 3, 9, 9, 20, 6, 92000)}
Anna 24
{'_id': ObjectId('69ae8dab85fca114abbd612a'), 'name': 'Lea', 'skills': ['C++', 'ML'], 'alter': 26, 'isStudent': False, 'eingeschrieben_am': datetime.datetime(2026, 3, 9, 9, 20, 6, 92000)}
Lea 26
{'_id': ObjectId('69c0ec0fb31cfe390202dd14'), 'name': 'Anna', 'alter': 24, 'isStudent': True}
Anna 24
{'_id': ObjectId('69c0ec11b31cfe390202dd15'), 'name': 'Jonas', 'skills': ['Python', 'MongoDB'], 'alter': 24}
Jonas 24
{'_id': ObjectId('69c0ec11b31cfe390202dd16'), 'name': 'Lea', 'skills': ['C++', 'ML'], 'alter': 26}
Lea 26


#### Aufgabe 4: Dokumente ändern

In [5]:
col.update_many({'alter': {'$gt': 25}}, {'$set': {'isStudent': False}})

UpdateResult({'n': 2, 'nModified': 1, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)

#### Aufgabe 5: Löschen eines Dokuments

In [6]:
col.delete_one({'name': 'Jonas'})

DeleteResult({'n': 1, 'ok': 1.0}, acknowledged=True)

#### Aufgabe 6: Datumsfelder

In [7]:
import datetime
import dateutil

col.update_many({}, {'$set': {'eingeschrieben_am': datetime.datetime.utcnow()}})

UpdateResult({'n': 4, 'nModified': 4, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)

#### Aufgabe 7: Sortierung und Projektion

In [8]:
query = col.find().sort('alter', -1)

for doc in query:
    del doc['_id']
    print(doc)

{'name': 'Lea', 'skills': ['C++', 'ML'], 'alter': 26, 'isStudent': False, 'eingeschrieben_am': datetime.datetime(2026, 3, 23, 7, 30, 33, 180000)}
{'name': 'Lea', 'skills': ['C++', 'ML'], 'alter': 26, 'isStudent': False, 'eingeschrieben_am': datetime.datetime(2026, 3, 23, 7, 30, 33, 180000)}
{'name': 'Anna', 'alter': 24, 'isStudent': True, 'eingeschrieben_am': datetime.datetime(2026, 3, 23, 7, 30, 33, 180000)}
{'name': 'Anna', 'alter': 24, 'isStudent': True, 'eingeschrieben_am': datetime.datetime(2026, 3, 23, 7, 30, 33, 180000)}


#### Aufgabe 8: Komplexes Dokument mit eingebetteten Objekten

In [9]:
kurseCol = db['kurse']

kurseCol.insert_one({
    'name': 'MongoDb Basics',
    'dozent': {
        'name': 'Müller',
        'e-mail': 'mueller@ts-münchen.de'
    },
    'studenten': [
        {
            'name': 'Anna',
            'note': 1.7,
        },
        {
            'name': 'Lea',
            'note': 2.3
        }
    ]
})

InsertOneResult(ObjectId('69c0ec1cb31cfe390202dd17'), acknowledged=True)

#### Aufgabe 9: Suche in Arrays

In [10]:
query = kurseCol.find({'studenten': {'$elemMatch': {'note': {'$lt': 2.0}}}})
for doc in query:
    print(doc)

{'_id': ObjectId('69ae930985fca114abbd612b'), 'name': 'MongoDb Basics', 'dozent': {'name': 'Müller', 'e-mail': 'mueller@ts-münchen.de'}, 'studenten': [{'name': 'Anna', 'note': 1.7}, {'name': 'Lea', 'note': 2.3}]}
{'_id': ObjectId('69c0ec1cb31cfe390202dd17'), 'name': 'MongoDb Basics', 'dozent': {'name': 'Müller', 'e-mail': 'mueller@ts-münchen.de'}, 'studenten': [{'name': 'Anna', 'note': 1.7}, {'name': 'Lea', 'note': 2.3}]}


#### Aufgabe 10: Zählen und Gruppieren

In [ ]:
query = kurseCol.aggregate([{
    '$group': {
        '_id': '',
        'students': {
            '$count': 'isStudent'
        }
    }
}])
for doc in query:
    print(doc)

OperationFailure: The field '$isStudent' must be an accumulator object, full error: {'ok': 0.0, 'errmsg': "The field '$isStudent' must be an accumulator object", 'code': 40234, 'codeName': 'Location40234'}

#### Aufgabe 11: Arbeiten mit ObjectId

In [18]:
query = kurseCol.find({
    '_id': ObjectId('69ae930985fca114abbd612b')
})
for doc in query:
    print(doc)

{'_id': ObjectId('69ae930985fca114abbd612b'), 'name': 'MongoDb Basics', 'dozent': {'name': 'Müller', 'e-mail': 'mueller@ts-münchen.de'}, 'studenten': [{'name': 'Anna', 'note': 1.7}, {'name': 'Lea', 'note': 2.3}]}


#### Aufgabe 12: Null-Werte & Existenzprüfung

In [32]:
query = kurseCol.find({
    'phone' : 'null'
})
for doc in query:
    print(doc)

#query = col.update_many({
#    'phone' : 'null'
#},{
#    'phone' : '999'
#})

#### Aufgabe 13: Lookup

#### Expertenaufgabe 14: GeoJSON und Geospatial Query